<h1>PyArrow Functionality</h1>

<p>pandas can utilize <a href="https://arrow.apache.org/docs/python/index.html">PyArrow</a> to extend functionality and improve the performance of various APIs. This includes:</p>

<ul class="simple">
<li><p>More extensive <a href="https://arrow.apache.org/docs/python/api/datatypes.html">data types</a> compared to NumPy</p></li>
<li><p>Missing data support (NA) for all data types</p></li>
<li><p>Performant IO reader integration</p></li>
<li><p>Facilitate interoperability with other dataframe libraries based on the Apache Arrow specification (e.g. polars, cuDF)</p></li>
</ul>

<p>To use this functionality, please ensure you have
<a href="https://pandas.pydata.org/docs/getting_started/install.html#install-optional-dependencies">installed the minimum supported PyArrow version.</a></p>

# <h2>Data Structure Integration</h2>

<p>A <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a>,
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a>, or the columns of a
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html" title="pandas.DataFrame"><code>DataFrame</code></a> can be directly backed by a
<a href="https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.ChunkedArray</code></a> which is similar to a NumPy array.
To construct these from the main pandas data structures, you can pass in a string of the type followed by <code>[pyarrow]</code>, e.g. <code>"int64[pyarrow]""</code> into the <code>dtype</code> parameter</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell0"><span></span><span class="gp">In [1]: </span><span class="n">ser</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="o">-</span><span class="mf">1.5</span><span class="p">,</span> <span class="mf">0.2</span><span class="p">,</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="s2">"float32[pyarrow]"</span><span class="p">)</span>

<span class="gp">In [2]: </span><span class="n">ser</span>
<span class="gh">Out[2]: </span>
<span class="go">0    -1.5</span>
<span class="go">1     0.2</span>
<span class="go">2    &lt;NA&gt;</span>
<span class="go">dtype: float[pyarrow]</span>

<span class="gp">In [3]: </span><span class="n">idx</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Index</span><span class="p">([</span><span class="kc">True</span><span class="p">,</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="s2">"bool[pyarrow]"</span><span class="p">)</span>

<span class="gp">In [4]: </span><span class="n">idx</span>
<span class="gh">Out[4]: </span><span class="go">Index([True, &lt;NA&gt;], dtype='bool[pyarrow]')</span>

<span class="gp">In [5]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">([[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="p">[</span><span class="mi">3</span><span class="p">,</span> <span class="mi">4</span><span class="p">]],</span> <span class="n">dtype</span><span class="o">=</span><span class="s2">"uint64[pyarrow]"</span><span class="p">)</span>

<span class="gp">In [6]: </span><span class="n">df</span>
<span class="gh">Out[6]: </span>
<span class="go">   0  1</span>
<span class="go">0  1  2</span>
<span class="go">1  3  4</span>
</pre>
</div>
</div>

<div class="admonition note">
<p class="admonition-title">Note</p>
<p>The string alias <code>"string[pyarrow]"</code> maps to <code>pd.StringDtype("pyarrow")</code> which is not equivalent to specifying <code>dtype=pd.ArrowDtype(pa.string())</code>. Generally, operations on the data will behave similarly except <code>pd.StringDtype("pyarrow")</code> can return NumPy-backed nullable types while <code>pd.ArrowDtype(pa.string())</code> will return
<a href="https://pandas.pydata.org/docs/reference/api/pandas.ArrowDtype.html" title="pandas.ArrowDtype"><code>ArrowDtype</code></a>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell1"><span></span><span class="gp">In [7]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">pyarrow</span><span class="w"> </span><span class="k">as</span><span class="w"> </span><span class="nn">pa</span>

<span class="gp">In [8]: </span><span class="n">data</span> <span class="o">=</span> <span class="nb">list</span><span class="p">(</span><span class="s2">"abc"</span><span class="p">)</span>

<span class="gp">In [9]: </span><span class="n">ser_sd</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">data</span><span class="p">,</span> <span class="n">dtype</span><span class="o">=</span><span class="s2">"string[pyarrow]"</span><span class="p">)</span>

<span class="gp">In [10]: </span><span class="n">ser_ad</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">data</span><span class="p">,</span> <span class="n">dtype</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">string</span><span class="p">()))</span>

<span class="gp">In [11]: </span><span class="n">ser_ad</span><span class="o">.</span><span class="n">dtype</span> <span class="o">==</span> <span class="n">ser_sd</span><span class="o">.</span><span class="n">dtype</span>
<span class="gh">Out[11]: </span><span class="go">False</span>

<span class="gp">In [12]: </span><span class="n">ser_sd</span><span class="o">.</span><span class="n">str</span><span class="o">.</span><span class="n">contains</span><span class="p">(</span><span class="s2">"a"</span><span class="p">)</span>
<span class="gh">Out[12]: </span>
<span class="go">0     True</span>
<span class="go">1    False</span>
<span class="go">2    False</span>
<span class="go">dtype: boolean</span>

<span class="gp">In [13]: </span><span class="n">ser_ad</span><span class="o">.</span><span class="n">str</span><span class="o">.</span><span class="n">contains</span><span class="p">(</span><span class="s2">"a"</span><span class="p">)</span>
<span class="gh">Out[13]: </span>
<span class="go">0     True</span>
<span class="go">1    False</span>
<span class="go">2    False</span>
<span class="go">dtype: bool[pyarrow]</span>
</pre>
</div>
</div>
</div>

<p>For PyArrow types that accept parameters, you can pass in a PyArrow type with those parameters into
<a href="https://pandas.pydata.org/docs/reference/api/pandas.ArrowDtype.html" title="pandas.ArrowDtype"><code>ArrowDtype</code></a> to use in the <code>dtype</code> parameter.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell2"><span></span><span class="gp">In [14]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">pyarrow</span><span class="w"> </span><span class="k">as</span><span class="w"> </span><span class="nn">pa</span>

<span class="gp">In [15]: </span><span class="n">list_str_type</span> <span class="o">=</span> <span class="n">pa</span><span class="o">.</span><span class="n">list_</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">string</span><span class="p">())</span>

<span class="gp">In [16]: </span><span class="n">ser</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([[</span><span class="s2">"hello"</span><span class="p">],</span> <span class="p">[</span><span class="s2">"there"</span><span class="p">]],</span> <span class="n">dtype</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">(</span><span class="n">list_str_type</span><span class="p">))</span>

<span class="gp">In [17]: </span><span class="n">ser</span>
<span class="gh">Out[17]: </span>
<span class="go">0    ['hello']</span>
<span class="go">1    ['there']</span>
<span class="go">dtype: list&lt;item: string&gt;[pyarrow]</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell3"><span></span><span class="gp">In [18]: </span><span class="kn">from</span><span class="w"> </span><span class="nn">datetime</span><span class="w"> </span><span class="kn">import</span> <span class="n">time</span>

<span class="gp">In [19]: </span><span class="n">idx</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Index</span><span class="p">([</span><span class="n">time</span><span class="p">(</span><span class="mi">12</span><span class="p">,</span> <span class="mi">30</span><span class="p">),</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">time64</span><span class="p">(</span><span class="s2">"us"</span><span class="p">)))</span>

<span class="gp">In [20]: </span><span class="n">idx</span>
<span class="gh">Out[20]: </span><span class="go">Index([12:30:00, &lt;NA&gt;], dtype='time64[us][pyarrow]')</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell4"><span></span><span class="gp">In [21]: </span><span class="kn">from</span><span class="w"> </span><span class="nn">decimal</span><span class="w"> </span><span class="kn">import</span> <span class="n">Decimal</span>

<span class="gp">In [22]: </span><span class="n">decimal_type</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">decimal128</span><span class="p">(</span><span class="mi">3</span><span class="p">,</span> <span class="n">scale</span><span class="o">=</span><span class="mi">2</span><span class="p">))</span>

<span class="gp">In [23]: </span><span class="n">data</span> <span class="o">=</span> <span class="p">[[</span><span class="n">Decimal</span><span class="p">(</span><span class="s2">"3.19"</span><span class="p">),</span> <span class="kc">None</span><span class="p">],</span> <span class="p">[</span><span class="kc">None</span><span class="p">,</span> <span class="n">Decimal</span><span class="p">(</span><span class="s2">"-1.23"</span><span class="p">)]]</span>

<span class="gp">In [24]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">data</span><span class="p">,</span> <span class="n">dtype</span><span class="o">=</span><span class="n">decimal_type</span><span class="p">)</span>

<span class="gp">In [25]: </span><span class="n">df</span>
<span class="gh">Out[25]: </span>
<span class="go">      0      1</span>
<span class="go">0  3.19   &lt;NA&gt;</span>
<span class="go">1  &lt;NA&gt;  -1.23</span>
</pre>
</div>
</div>

<p>If you already have an <a href="https://arrow.apache.org/docs/python/generated/pyarrow.Array.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.Array</code></a> or <a href="https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.ChunkedArray</code></a>, you can pass it into
<a href="https://pandas.pydata.org/docs/reference/api/pandas.arrays.ArrowExtensionArray.html" title="pandas.arrays.ArrowExtensionArray"><code>arrays.ArrowExtensionArray</code></a> to construct the associated <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a>, <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a> or
<a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html" title="pandas.DataFrame"><code>DataFrame</span></code></a> object.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell5"><span></span><span class="gp">In [26]: </span><span class="n">pa_array</span> <span class="o">=</span> <span class="n">pa</span><span class="o">.</span><span class="n">array</span><span class="p">(</span>
<span class="gp">   ....: </span>    <span class="p">[{</span><span class="s2">"1"</span><span class="p">:</span> <span class="s2">"2"</span><span class="p">},</span> <span class="p">{</span><span class="s2">"10"</span><span class="p">:</span> <span class="s2">"20"</span><span class="p">},</span> <span class="kc">None</span><span class="p">],</span>
<span class="gp">   ....: </span>    <span class="nb">type</span><span class="o">=</span><span class="n">pa</span><span class="o">.</span><span class="n">map_</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">string</span><span class="p">(),</span> <span class="n">pa</span><span class="o">.</span><span class="n">string</span><span class="p">()),</span>
<span class="gp">   ....: </span><span class="p">)</span>
<span class="gp">   ....: </span>

<span class="gp">In [27]: </span><span class="n">ser</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">pd</span><span class="o">.</span><span class="n">arrays</span><span class="o">.</span><span class="n">ArrowExtensionArray</span><span class="p">(</span><span class="n">pa_array</span><span class="p">))</span>

<span class="gp">In [28]: </span><span class="n">ser</span>
<span class="gh">Out[28]: </span>
<span class="go">0      [('1', '2')]</span>
<span class="go">1    [('10', '20')]</span>
<span class="go">2              &lt;NA&gt;</span>
<span class="go">dtype: map&lt;string, string&gt;[pyarrow]</span>
</pre>
</div>
</div>

<p>To retrieve a pyarrow <a href="https://arrow.apache.org/docs/python/generated/pyarrow.ChunkedArray.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.ChunkedArray</code></a> from a <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a> or <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a>, you can call the pyarrow array constructor on the <a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.html" title="pandas.Series"><code>Series</code></a> or <a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html" title="pandas.Index"><code>Index</code></a>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell6"><span></span><span class="gp">In [29]: </span><span class="n">ser</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="s2">"uint8[pyarrow]"</span><span class="p">)</span>

<span class="gp">In [30]: </span><span class="n">pa</span><span class="o">.</span><span class="n">array</span><span class="p">(</span><span class="n">ser</span><span class="p">)</span>
<span class="gh">Out[30]: </span>
<span class="go">&lt;pyarrow.lib.UInt8Array object at 0x7f842e072860&gt;</span>
<span class="go">[</span>
<span class="go">  1,</span>
<span class="go">  2,</span>
<span class="go">  null</span>
<span class="go">]</span>

<span class="gp">In [31]: </span><span class="n">idx</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Index</span><span class="p">(</span><span class="n">ser</span><span class="p">)</span>

<span class="gp">In [32]: </span><span class="n">pa</span><span class="o">.</span><span class="n">array</span><span class="p">(</span><span class="n">idx</span><span class="p">)</span>
<span class="gh">Out[32]: </span>
<span class="go">&lt;pyarrow.lib.UInt8Array object at 0x7f842e0726e0&gt;</span>
<span class="go">[</span>
<span class="go">  1,</span>
<span class="go">  2,</span>
<span class="go">  null</span>
<span class="go">]</span>
</pre>
</div>
</div>

<p>To convert a <a href="https://arrow.apache.org/docs/python/generated/pyarrow.Table.html" title="(in Apache Arrow v21.0.0)"><code>pyarrow.Table</code></a> to a <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html" title="pandas.DataFrame"><code>DataFrame</code></a>, you can call the
<a href="https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pandas" title="(in Apache Arrow v21.0.0)"><code>pyarrow.Table.to_pandas()</code></a> method with <code>types_mapper=pd.ArrowDtype</code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell7"><span></span><span class="gp">In [33]: </span><span class="n">table</span> <span class="o">=</span> <span class="n">pa</span><span class="o">.</span><span class="n">table</span><span class="p">([</span><span class="n">pa</span><span class="o">.</span><span class="n">array</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">,</span> <span class="mi">3</span><span class="p">],</span> <span class="nb">type</span><span class="o">=</span><span class="n">pa</span><span class="o">.</span><span class="n">int64</span><span class="p">())],</span> <span class="n">names</span><span class="o">=</span><span class="p">[</span><span class="s2">"a"</span><span class="p">])</span>

<span class="gp">In [34]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">table</span><span class="o">.</span><span class="n">to_pandas</span><span class="p">(</span><span class="n">types_mapper</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">)</span>

<span class="gp">In [35]: </span><span class="n">df</span>
<span class="gh">Out[35]: </span>
<span class="go">   a</span>
<span class="go">0  1</span>
<span class="go">1  2</span>
<span class="go">2  3</span>

<span class="gp">In [36]: </span><span class="n">df</span><span class="o">.</span><span class="n">dtypes</span>
<span class="gh">Out[36]: </span>
<span class="go">a    int64[pyarrow]</span>
<span class="go">dtype: object</span>
</pre>
</div>
</div>

# <h2>Operations</h2>

<p>PyArrow data structure integration is implemented through pandas’
<a href="https://pandas.pydata.org/docs/reference/api/pandas.api.extensions.ExtensionArray.html" title="pandas.api.extensions.ExtensionArray"><code>ExtensionArray</span></code></a> <a href="https://pandas.pydata.org/docs/development/extending.html#extending-extension-type">interface</span></a>; therefore, supported functionality exists where this interface is integrated within the pandas API. Additionally, this functionality is accelerated with PyArrow <a href="https://arrow.apache.org/docs/python/api/compute.html">compute functions</a> where available. This includes:</p>

<ul class="simple">
<li><p>Numeric aggregations</p></li>
<li><p>Numeric arithmetic</p></li>
<li><p>Numeric rounding</p></li>
<li><p>Logical and comparison functions</p></li>
<li><p>String functionality</p></li>
<li><p>Datetime functionality</p></li>
</ul>

<p>The following are just some examples of operations that are accelerated by native PyArrow compute functions.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell8"><span></span><span class="gp">In [37]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">pyarrow</span><span class="w"> </span><span class="k">as</span><span class="w"> </span><span class="nn">pa</span>

<span class="gp">In [38]: </span><span class="n">ser</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="o">-</span><span class="mf">1.545</span><span class="p">,</span> <span class="mf">0.211</span><span class="p">,</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="s2">"float32[pyarrow]"</span><span class="p">)</span>

<span class="gp">In [39]: </span><span class="n">ser</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>
<span class="gh">Out[39]: </span><span class="go">-0.6669999808073044</span>

<span class="gp">In [40]: </span><span class="n">ser</span> <span class="o">+</span> <span class="n">ser</span>
<span class="gh">Out[40]: </span>
<span class="go">0    -3.09</span>
<span class="go">1    0.422</span>
<span class="go">2     &lt;NA&gt;</span>
<span class="go">dtype: float[pyarrow]</span>

<span class="gp">In [41]: </span><span class="n">ser</span> <span class="o">&gt;</span> <span class="p">(</span><span class="n">ser</span> <span class="o">+</span> <span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[41]: </span>
<span class="go">0    False</span>
<span class="go">1    False</span>
<span class="go">2     &lt;NA&gt;</span>
<span class="go">dtype: bool[pyarrow]</span>

<span class="gp">In [42]: </span><span class="n">ser</span><span class="o">.</span><span class="n">dropna</span><span class="p">()</span>
<span class="gh">Out[42]: </span>
<span class="go">0   -1.545</span>
<span class="go">1    0.211</span>
<span class="go">dtype: float[pyarrow]</span>

<span class="gp">In [43]: </span><span class="n">ser</span><span class="o">.</span><span class="n">isna</span><span class="p">()</span>
<span class="gh">Out[43]: </span>
<span class="go">0    False</span>
<span class="go">1    False</span>
<span class="go">2     True</span>
<span class="go">dtype: bool</span>

<span class="gp">In [44]: </span><span class="n">ser</span><span class="o">.</span><span class="n">fillna</span><span class="p">(</span><span class="mi">0</span><span class="p">)</span>
<span class="gh">Out[44]: </span>
<span class="go">0   -1.545</span>
<span class="go">1    0.211</span>
<span class="go">2      0.0</span>
<span class="go">dtype: float[pyarrow]</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell9"><span></span><span class="gp">In [45]: </span><span class="n">ser_str</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"b"</span><span class="p">,</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">string</span><span class="p">()))</span>

<span class="gp">In [46]: </span><span class="n">ser_str</span><span class="o">.</span><span class="n">str</span><span class="o">.</span><span class="n">startswith</span><span class="p">(</span><span class="s2">"a"</span><span class="p">)</span>
<span class="gh">Out[46]: </span>
<span class="go">0     True</span>
<span class="go">1    False</span>
<span class="go">2     &lt;NA&gt;</span>
<span class="go">dtype: bool[pyarrow]</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell10"><span></span><span class="gp">In [47]: </span><span class="kn">from</span><span class="w"> </span><span class="nn">datetime</span><span class="w"> </span><span class="kn">import</span> <span class="n">datetime</span>

<span class="gp">In [48]: </span><span class="n">pa_type</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">ArrowDtype</span><span class="p">(</span><span class="n">pa</span><span class="o">.</span><span class="n">timestamp</span><span class="p">(</span><span class="s2">"ns"</span><span class="p">))</span>

<span class="gp">In [49]: </span><span class="n">ser_dt</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">([</span><span class="n">datetime</span><span class="p">(</span><span class="mi">2022</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">),</span> <span class="kc">None</span><span class="p">],</span> <span class="n">dtype</span><span class="o">=</span><span class="n">pa_type</span><span class="p">)</span>

<span class="gp">In [50]: </span><span class="n">ser_dt</span><span class="o">.</span><span class="n">dt</span><span class="o">.</span><span class="n">strftime</span><span class="p">(</span><span class="s2">"%Y-%m"</span><span class="p">)</span>
<span class="gh">Out[50]: </span>
<span class="go">0    2022-01</span>
<span class="go">1       &lt;NA&gt;</span>
<span class="go">dtype: string[pyarrow]</span>
</pre>
</div>
</div>

# <h2>I/O Reading</h2>

<p>PyArrow also provides IO reading functionality that has been integrated into several pandas IO readers. The following functions provide an <code>engine</span></code> keyword that can dispatch to PyArrow to accelerate reading from an IO source.</p>

<ul class="simple">
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html" title="pandas.read_csv"><code>read_csv()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_json.html" title="pandas.read_json"><code>read_json()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_orc.html" title="pandas.read_orc"><code>read_orc()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.read_feather.html" title="pandas.read_feather"><code>read_feather()</code></a></p></li>
</ul>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell11"><span></span><span class="gp">In [51]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">io</span>

<span class="gp">In [52]: </span><span class="n">data</span> <span class="o">=</span> <span class="n">io</span><span class="o">.</span><span class="n">StringIO</span><span class="p">(</span><span class="s2">"""a,b,c</span>
<span class="gp">   ....: </span><span class="s2">   1,2.5,True</span>
<span class="gp">   ....: </span><span class="s2">   3,4.5,False</span>
<span class="gp">   ....: </span><span class="s2">"""</span><span class="p">)</span>
<span class="gp">   ....: </span>

<span class="gp">In [53]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">read_csv</span><span class="p">(</span><span class="n">data</span><span class="p">,</span> <span class="n">engine</span><span class="o">=</span><span class="s2">"pyarrow"</span><span class="p">)</span>

<span class="gp">In [54]: </span><span class="n">df</span>
<span class="gh">Out[54]: </span>
<span class="go">   a    b      c</span>
<span class="go">0  1  2.5   True</span>
<span class="go">1  3  4.5  False</span>
</pre>
</div>
</div>

<p>By default, these functions and all other IO reader functions return NumPy-backed data. These readers can return PyArrow-backed data by specifying the parameter <code>dtype_backend="pyarrow"</code>.
A reader does not need to set <code>engine="pyarrow"</span></code> to necessarily return PyArrow-backed data.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell12"><span></span><span class="gp">In [55]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">io</span>

<span class="gp">In [56]: </span><span class="n">data</span> <span class="o">=</span> <span class="n">io</span><span class="o">.</span><span class="n">StringIO</span><span class="p">(</span><span class="s2">"""a,b,c,d,e,f,g,h,i</span>
<span class="gp">   ....: </span><span class="s2">    1,2.5,True,a,,,,,</span>
<span class="gp">   ....: </span><span class="s2">    3,4.5,False,b,6,7.5,True,a,</span>
<span class="gp">   ....: </span><span class="s2">"""</span><span class="p">)</span>
<span class="gp">   ....: </span>

<span class="gp">In [57]: </span><span class="n">df_pyarrow</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">read_csv</span><span class="p">(</span><span class="n">data</span><span class="p">,</span> <span class="n">dtype_backend</span><span class="o">=</span><span class="s2">"pyarrow"</span><span class="p">)</span>

<span class="gp">In [58]: </span><span class="n">df_pyarrow</span><span class="o">.</span><span class="n">dtypes</span>
<span class="gh">Out[58]: </span>
<span class="go">a     int64[pyarrow]</span>
<span class="go">b    double[pyarrow]</span>
<span class="go">c      bool[pyarrow]</span>
<span class="go">d    string[pyarrow]</span>
<span class="go">e     int64[pyarrow]</span>
<span class="go">f    double[pyarrow]</span>
<span class="go">g      bool[pyarrow]</span>
<span class="go">h    string[pyarrow]</span>
<span class="go">i      null[pyarrow]</span>
<span class="go">dtype: object</span>
</pre>
</div>
</div>

<p>Several non-IO reader functions can also use the <code>dtype_backend</code> argument to return PyArrow-backed data including:</p>

<ul class="simple">
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html" title="pandas.to_numeric"><code>to_numeric()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html" title="pandas.DataFrame.convert_dtypes"><code>DataFrame.convert_dtypes()</code></a></p></li>
<li><p><a href="https://pandas.pydata.org/docs/reference/api/pandas.Series.convert_dtypes.html" title="pandas.Series.convert_dtypes"><code>Series.convert_dtypes()</code></a></p></li>
</ul>